In [1]:
import os
import sys
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
os.chdir(PROJECT_ROOT)

# Make configs/ importable
sys.path.insert(0, str(PROJECT_ROOT))
from configs.seed import SEED
print(f"Using SEED = {SEED}")

con = duckdb.connect(str(DATA_DIR / "criteo.duckdb"))
print(f"Connected. Row count: {con.execute('SELECT COUNT(*) FROM criteo').fetchone()[0]:,}")

Using SEED = 42
Connected. Row count: 13,979,592


## Subsampling decision: 14M → 2M rows

**Why 2M:**

1. **Preserves rare-event signal.** At 0.19% control conversion rate,
   2M rows still yield ~3,800 control conversions — enough for stable
   per-bucket and per-decile estimates downstream.
2. **Fits in 16GB RAM.** Full 14M × 12 float64 features ≈ 1.3 GB just
   for X, before treatment/outcome arrays, model copies, and prediction
   buffers. 2M keeps total memory under 4 GB during training.
3. **Iteration speed.** LightGBM T-learner training time scales roughly
   linearly above 1M rows. 2M targets a 2–5 minute fit, fast enough to
   iterate on hyperparameters; 14M would push to 20+ minutes per fit.

**Stratification key:** `(treatment, conversion)` ∈ {(0,0), (0,1), (1,0), (1,1)}.

This is the only meaningful stratification for uplift modeling. Visit
isn't used because it's correlated with conversion (most converters
visited first), so stratifying on conversion already preserves the
visit distribution within tolerance. We verify this after sampling.

**Sampling fraction:** 2,000,000 / 13,979,592 = **0.1431** (14.31%).

Each stratum gets sampled at this same fraction, which preserves all
relative proportions: 85/15 treatment ratio, 0.19%/0.31% control/
treated conversion rates, ~0.04 visit rate. Documented checks below.

In [3]:
TARGET_N = 2_000_000

# Get original stratum sizes so we can sample proportionally
strata = con.execute("""
    SELECT treatment, conversion, COUNT(*) AS n
    FROM criteo
    GROUP BY treatment, conversion
    ORDER BY treatment, conversion
""").fetchdf()
strata["pct"] = strata["n"] / strata["n"].sum()
strata["target_n"] = (strata["pct"] * TARGET_N).round().astype(int)
print("Stratum sizes (original → target):")
print(strata)
print(f"\nTotal target: {strata['target_n'].sum():,}")

Stratum sizes (original → target):
   treatment  conversion         n       pct  target_n
0          0           0   2092874  0.149709    299418
1          0           1      4063  0.000291       581
2          1           0  11845944  0.847374   1694748
3          1           1     36711  0.002626      5252

Total target: 1,999,999


In [4]:
%%time
# Use a deterministic ROW_NUMBER ordering with hash-based randomization,
# so the seed is reproducible across DuckDB versions.
con.execute(f"DROP TABLE IF EXISTS criteo_sample")

con.execute(f"""
CREATE TABLE criteo_sample AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY treatment, conversion
            ORDER BY hash(rowid + {SEED})
        ) AS row_in_stratum,
        COUNT(*) OVER (PARTITION BY treatment, conversion) AS stratum_size
    FROM criteo
)
SELECT * EXCLUDE (row_in_stratum, stratum_size)
FROM ranked
WHERE row_in_stratum <= CAST(stratum_size * ({TARGET_N} * 1.0 / {strata['n'].sum()}) AS BIGINT)
""")

n_sample = con.execute("SELECT COUNT(*) FROM criteo_sample").fetchone()[0]
print(f"Sampled {n_sample:,} rows  (target was {TARGET_N:,})")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sampled 1,999,999 rows  (target was 2,000,000)
CPU times: user 13.2 s, sys: 4.64 s, total: 17.9 s
Wall time: 3.7 s


In [5]:
# Total rows
n_sample = con.execute("SELECT COUNT(*) FROM criteo_sample").fetchone()[0]

# Treatment ratio
tx = con.execute("""
    SELECT treatment, COUNT(*) AS n,
           ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (), 4) AS pct
    FROM criteo_sample GROUP BY treatment ORDER BY treatment
""").fetchdf()

# Conversion rate by treatment
cv = con.execute("""
    SELECT treatment, AVG(conversion) AS conv_rate, AVG(visit) AS visit_rate, COUNT(*) AS n
    FROM criteo_sample GROUP BY treatment ORDER BY treatment
""").fetchdf()

# Compare to original (from Day 4 numbers)
print(f"Sample size: {n_sample:,}")
print(f"\nTreatment split:\n{tx}\n")
print(f"Outcome rates by treatment:\n{cv}\n")

print("Expected (from Day 4 on full data):")
print("  treatment=0: conv_rate=0.001938, visit_rate=0.038201")
print("  treatment=1: conv_rate=0.003089, visit_rate=0.048543")

Sample size: 1,999,999

Treatment split:
   treatment        n   pct
0          0   299999  0.15
1          1  1700000  0.85

Outcome rates by treatment:
   treatment  conv_rate  visit_rate        n
0          0   0.001937    0.038403   299999
1          1   0.003089    0.048439  1700000

Expected (from Day 4 on full data):
  treatment=0: conv_rate=0.001938, visit_rate=0.038201
  treatment=1: conv_rate=0.003089, visit_rate=0.048543


In [6]:
%%time
sample_df = con.execute("SELECT *, ROW_NUMBER() OVER () AS row_id FROM criteo_sample").fetchdf()
# row_id will be useful in Day 15 for joining predictions back to SQL
print(f"Loaded {len(sample_df):,} rows, memory: {sample_df.memory_usage(deep=True).sum()/1e6:.1f} MB")
sample_df.head(3)

Loaded 1,999,999 rows, memory: 272.0 MB
CPU times: user 71.6 ms, sys: 95.6 ms, total: 167 ms
Wall time: 182 ms


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure,row_id
0,13.074097,10.059654,8.293415,-2.118394,11.029584,3.013064,-17.999322,10.257657,3.860579,25.240993,6.076543,-0.168679,0,1,1,0,1
1,12.616365,10.059654,8.521277,4.679882,11.029584,4.115453,0.294443,4.833815,3.834451,31.796212,5.741455,-0.267350,0,1,1,0,2
2,23.829714,10.059654,8.629945,4.679882,14.439578,4.115453,-11.495164,4.833815,3.782140,41.439237,5.882683,-0.624187,0,1,1,0,3


In [7]:
from sklearn.model_selection import StratifiedShuffleSplit

# Build the 4-way stratification key
strat_key = sample_df["treatment"].astype(str) + "_" + sample_df["conversion"].astype(str)
print("Strata composition:")
print(strat_key.value_counts())

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(sample_df, strat_key))

train_df = sample_df.iloc[train_idx].reset_index(drop=True)
test_df  = sample_df.iloc[test_idx].reset_index(drop=True)

print(f"\nTrain: {len(train_df):,}  Test: {len(test_df):,}")
print(f"Ratio: {len(train_df)/len(sample_df):.2%} / {len(test_df)/len(sample_df):.2%}")

Strata composition:
1_0    1694748
0_0     299418
1_1       5252
0_1        581
Name: count, dtype: int64

Train: 1,599,999  Test: 400,000
Ratio: 80.00% / 20.00%


In [9]:
def split_diagnostics(df, label):
    n = len(df)
    treat_pct = df["treatment"].mean()
    conv_rate = df["conversion"].mean()
    visit_rate = df["visit"].mean()
    conv_t = df.loc[df.treatment==1, "conversion"].mean()
    conv_c = df.loc[df.treatment==0, "conversion"].mean()
    print(f"{label:8s}  n={n:>10,}  treat_pct={treat_pct:.4f}  "
          f"conv={conv_rate:.6f}  visit={visit_rate:.6f}  "
          f"conv_t={conv_t:.6f}  conv_c={conv_c:.6f}")

split_diagnostics(sample_df, "sample")
split_diagnostics(train_df,  "train")
split_diagnostics(test_df,   "test")

sample    n= 1,999,999  treat_pct=0.8500  conv=0.002917  visit=0.046934  conv_t=0.003089  conv_c=0.001937
train     n= 1,599,999  treat_pct=0.8500  conv=0.002917  visit=0.046909  conv_t=0.003090  conv_c=0.001938
test      n=   400,000  treat_pct=0.8500  conv=0.002915  visit=0.047030  conv_t=0.003088  conv_c=0.001933


In [10]:
con.execute("DROP TABLE IF EXISTS train")
con.execute("DROP TABLE IF EXISTS test")

# DuckDB can register pandas DataFrames directly via the connection
con.register("train_df_view", train_df)
con.register("test_df_view",  test_df)

con.execute("CREATE TABLE train AS SELECT * FROM train_df_view")
con.execute("CREATE TABLE test  AS SELECT * FROM test_df_view")

con.unregister("train_df_view")
con.unregister("test_df_view")

# Verify
n_train = con.execute("SELECT COUNT(*) FROM train").fetchone()[0]
n_test  = con.execute("SELECT COUNT(*) FROM test").fetchone()[0]
print(f"Saved to DuckDB. train={n_train:,}, test={n_test:,}")

Saved to DuckDB. train=1,599,999, test=400,000


In [11]:
con.close()
print("Connection closed.")

Connection closed.


In [12]:
import os
import sys
import duckdb
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))
from configs.seed import SEED

con = duckdb.connect(str(DATA_DIR / "criteo.duckdb"))
n_train = con.execute("SELECT COUNT(*) FROM train").fetchone()[0]
n_test  = con.execute("SELECT COUNT(*) FROM test").fetchone()[0]
print(f"Connected. train={n_train:,}, test={n_test:,}")

Connected. train=1,599,999, test=400,000


In [13]:
%%time
train_df = con.execute("SELECT * FROM train").fetchdf()
test_df  = con.execute("SELECT * FROM test").fetchdf()
print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(f"Train memory: {train_df.memory_usage(deep=True).sum()/1e6:.1f} MB")

Train: (1599999, 17), Test: (400000, 17)
Train memory: 217.6 MB
CPU times: user 76.8 ms, sys: 91 ms, total: 168 ms
Wall time: 200 ms


In [14]:
feature_cols = [f"f{i}" for i in range(12)]

print("Train dtypes:")
print(train_df[feature_cols].dtypes)
print()

print(f"Train NaN counts per feature:")
print(train_df[feature_cols].isna().sum())
print(f"\nTest NaN counts per feature:")
print(test_df[feature_cols].isna().sum())

Train dtypes:
f0     float64
f1     float64
f2     float64
f3     float64
f4     float64
f5     float64
f6     float64
f7     float64
f8     float64
f9     float64
f10    float64
f11    float64
dtype: object

Train NaN counts per feature:
f0     0
f1     0
f2     0
f3     0
f4     0
f5     0
f6     0
f7     0
f8     0
f9     0
f10    0
f11    0
dtype: int64

Test NaN counts per feature:
f0     0
f1     0
f2     0
f3     0
f4     0
f5     0
f6     0
f7     0
f8     0
f9     0
f10    0
f11    0
dtype: int64


In [15]:
train_df[feature_cols] = train_df[feature_cols].astype("float32")
test_df[feature_cols]  = test_df[feature_cols].astype("float32")

print(f"Train memory after float32: {train_df.memory_usage(deep=True).sum()/1e6:.1f} MB")

Train memory after float32: 140.8 MB


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols]).astype("float32")
X_test  = scaler.transform(test_df[feature_cols]).astype("float32")

print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"X_test shape:  {X_test.shape}, dtype: {X_test.dtype}")
print(f"\nTrain scaler mean (should be ~0 after transform):")
print(f"  X_train.mean(axis=0)[:3] = {X_train.mean(axis=0)[:3]}")
print(f"\nTest scaler mean (should be ~0 but not exactly 0 — same shift as train):")
print(f"  X_test.mean(axis=0)[:3]  = {X_test.mean(axis=0)[:3]}")

X_train shape: (1599999, 12), dtype: float32
X_test shape:  (400000, 12), dtype: float32

Train scaler mean (should be ~0 after transform):
  X_train.mean(axis=0)[:3] = [-7.9984716e-08  4.1371181e-06 -7.0465131e-07]

Test scaler mean (should be ~0 but not exactly 0 — same shift as train):
  X_test.mean(axis=0)[:3]  = [-0.00445523 -0.00095783  0.00393576]


In [17]:
W_train = train_df["treatment"].to_numpy(dtype="int8")
W_test  = test_df["treatment"].to_numpy(dtype="int8")

y_train_visit = train_df["visit"].to_numpy(dtype="int8")
y_test_visit  = test_df["visit"].to_numpy(dtype="int8")

y_train_conv  = train_df["conversion"].to_numpy(dtype="int8")
y_test_conv   = test_df["conversion"].to_numpy(dtype="int8")

# Keep row_ids — Day 15 needs them to join predictions back to SQL
row_id_train = train_df["row_id"].to_numpy(dtype="int64")
row_id_test  = test_df["row_id"].to_numpy(dtype="int64")

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"W_train: {W_train.shape}, W_test: {W_test.shape}")
print(f"y_train_conv: {y_train_conv.shape} ({y_train_conv.sum()} positives)")
print(f"y_test_conv:  {y_test_conv.shape} ({y_test_conv.sum()} positives)")
print(f"y_train_visit: {y_train_visit.shape} ({y_train_visit.sum()} positives)")
print(f"y_test_visit:  {y_test_visit.shape} ({y_test_visit.sum()} positives)")

X_train: (1599999, 12), X_test: (400000, 12)
W_train: (1599999,), W_test: (400000,)
y_train_conv: (1599999,) (4667 positives)
y_test_conv:  (400000,) (1166 positives)
y_train_visit: (1599999,) (75055 positives)
y_test_visit:  (400000,) (18812 positives)


In [18]:
artifacts = {
    "X_train.pkl":      X_train,
    "X_test.pkl":       X_test,
    "W_train.pkl":      W_train,
    "W_test.pkl":       W_test,
    "y_train_conv.pkl": y_train_conv,
    "y_test_conv.pkl":  y_test_conv,
    "y_train_visit.pkl": y_train_visit,
    "y_test_visit.pkl":  y_test_visit,
    "row_id_train.pkl": row_id_train,
    "row_id_test.pkl":  row_id_test,
    "scaler.pkl":       scaler,
    "feature_cols.pkl": feature_cols,
}

for fname, obj in artifacts.items():
    with open(PROCESSED_DIR / fname, "wb") as f:
        pickle.dump(obj, f)

# Verify file sizes
import os
print(f"Wrote {len(artifacts)} files to {PROCESSED_DIR}:")
for fname in artifacts:
    size_mb = (PROCESSED_DIR / fname).stat().st_size / 1e6
    print(f"  {fname:25s}  {size_mb:>7.1f} MB")

Wrote 12 files to /Users/madhumithakatam/Documents/Projects/CriteoUpliftModeling/criteo_uplift_modeling/data/processed:
  X_train.pkl                   76.8 MB
  X_test.pkl                    19.2 MB
  W_train.pkl                    1.6 MB
  W_test.pkl                     0.4 MB
  y_train_conv.pkl               1.6 MB
  y_test_conv.pkl                0.4 MB
  y_train_visit.pkl              1.6 MB
  y_test_visit.pkl               0.4 MB
  row_id_train.pkl              12.8 MB
  row_id_test.pkl                3.2 MB
  scaler.pkl                     0.0 MB
  feature_cols.pkl               0.0 MB


In [19]:
# Sanity: load one back and confirm it's the same
with open(PROCESSED_DIR / "X_train.pkl", "rb") as f:
    X_train_reloaded = pickle.load(f)
assert np.allclose(X_train, X_train_reloaded), "Reload mismatch!"
assert X_train_reloaded.dtype == np.float32
print("Pickle round-trip OK.")

Pickle round-trip OK.


In [20]:
con.close()
print("Connection closed.")

Connection closed.


In [1]:
import os
import sys
from pathlib import Path

# Same setup as before — fresh kernel needs it
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_processed_data

data = load_processed_data()

# Verify shapes
expected_train_n = 1_600_000  # 80% of 2M, ±a few from int rounding
expected_test_n  = 400_000

assert data["X_train"].shape[1] == 12, f"Got {data['X_train'].shape[1]} features"
assert data["X_test"].shape[1] == 12
assert abs(data["X_train"].shape[0] - expected_train_n) < 1000
assert abs(data["X_test"].shape[0] - expected_test_n) < 1000
assert data["X_train"].dtype.name == "float32"
assert data["W_train"].dtype.name == "int8"

# Verify the scaler is what we expect
from sklearn.preprocessing import StandardScaler
assert isinstance(data["scaler"], StandardScaler)
assert hasattr(data["scaler"], "mean_"), "Scaler isn't fitted!"

print("Smoke test passed. Available keys:")
for k in data:
    v = data[k]
    if hasattr(v, "shape"):
        print(f"  {k:25s} shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k:25s} {type(v).__name__}")

Smoke test passed. Available keys:
  X_train                   shape=(1599999, 12), dtype=float32
  X_test                    shape=(400000, 12), dtype=float32
  W_train                   shape=(1599999,), dtype=int8
  W_test                    shape=(400000,), dtype=int8
  y_train_conv              shape=(1599999,), dtype=int8
  y_test_conv               shape=(400000,), dtype=int8
  y_train_visit             shape=(1599999,), dtype=int8
  y_test_visit              shape=(400000,), dtype=int8
  row_id_train              shape=(1599999,), dtype=int64
  row_id_test               shape=(400000,), dtype=int64
  scaler                    StandardScaler
  feature_cols              list
